In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Column
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime as datum_vreme


In [0]:
checkpoint_path = "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/openmeteo"

In [0]:
bronze_openmeteo = (spark.readStream.table("bg_traffic.bg_traffic_bronze.openmeteo"))


### Flattering

In [0]:
flatten = bronze_openmeteo.select(
    F.col("data.current.time").alias("weather_timestamp"),
    F.col("data.current.temperature_2m").alias("temperature_c"),
    F.col("data.current.relative_humidity_2m").alias("relative_humidity"),
    F.col("data.current.precipitation").alias("precipitation_mm"),
    F.col("data.current.wind_speed_10m").alias("wind_speed_kmh"),
    F.col("data.current.rain").alias("rain_mm"),
    F.col("fetched_at"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file")

    

    
)

In [0]:
flatten.printSchema()

### Casting

In [0]:
types = (flatten
    .withColumn("weather_timestamp", F.to_timestamp("weather_timestamp"))
    .withColumn("temperature_c",F.col("temperature_c").cast("double"))
    .withColumn("relative_humidity",F.col("relative_humidity").cast("double"))
    .withColumn("precipitation_mm",F.col("precipitation_mm").cast("double"))
    .withColumn("wind_speed_kmh",F.col("wind_speed_kmh").cast("double"))
    .withColumn("rain_mm",F.col("rain_mm").cast("double"))
    .withColumn("fetched_at", F.to_timestamp("fetched_at"))
)

types.printSchema()

### Dedup

In [0]:
dedup = (
    types.withWatermark("weather_timestamp","2 hours").dropDuplicates(['weather_timestamp'])
)

### Validate

In [0]:
valid = dedup.filter(
    (F.col("temperature_c").between(-30,50)) &
    (F.col("relative_humidity").between(0,100)) &
    (F.col("weather_timestamp").isNotNull()) &
    (F.col("fetched_at").isNotNull()) &
    (F.col("_ingestion_timestamp").isNotNull()) &
    (F.col("precipitation_mm") >= 0) &
    (F.col("rain_mm") >= 0) &
    (F.col("wind_speed_kmh") >= 0) &
    (F.col("rain_mm") <= F.col("precipitation_mm"))

    
    
    
    )




### Merge

In [0]:
def merge_to_silver(df_source, batch_id):
    silver_table = DeltaTable.forName(spark, "bg_traffic.bg_traffic_silver.openmeteo")
    (silver_table.alias("target").merge(df_source.alias("source"),"target.weather_timestamp = source.weather_timestamp")).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#### Write to Silver table

In [0]:
query = (
    valid.writeStream.foreachBatch(merge_to_silver).option("checkpointLocation",checkpoint_path).trigger(availableNow=True).start()
)

query.awaitTermination()

In [0]:
%sql
use catalog `bg_traffic`; select * from `bg_traffic_silver`.`openmeteo`;

In [0]:
bronze_table = spark.read.table("bg_traffic.bg_traffic_bronze.openmeteo")
bronze_table.display()

In [0]:
silver_table = spark.read.table("bg_traffic.bg_traffic_silver.openmeteo")
silver_table.display()